# Compare dates with and without human review

Configure the input paths in the setup cell, then run the cells in order. See the project README for input schemas and model setup.


In [ ]:
from pathlib import Path
import sys

# Locate the project when Jupyter starts in a notebook subfolder.
project_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "smmnlp" / "__init__.py").is_file()),
    None,
)
if project_root is None:
    raise RuntimeError("Start Jupyter inside SMMnlp_v2 or set its working directory there.")
sys.path.insert(0, str(project_root))


In [ ]:
import logging

import numpy as np
import pandas as pd

from smmnlp.paths import ProjectPaths
from smmnlp.parsing import expand_partial_date

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
paths = ProjectPaths.from_environment()
# Override paths here, or set SMMNLP_DATA_ROOT and SMMNLP_MODEL_PATH.

## Phase 3 results: SMM DxDate Training yes SMM, hx n=77 Testing  yes SMM, hx n=37


## Training & Testing ids


In [ ]:
readfiletrue = paths.data("Update/True/SMMNLP_training_true_phase1.xlsx")
true_training = pd.read_excel(readfiletrue)

In [ ]:
readfiletrue = paths.data("Update/True/SMMNLP_testing_true_phase1.xlsx")
true_testing = pd.read_excel(readfiletrue)

In [ ]:
train_ids = true_training["PatientSSN"].unique()
test_ids = true_testing["PatientSSN"].unique()

In [ ]:
len(train_ids)

In [ ]:
len(test_ids)

## Read in true labels


In [ ]:
truedata = paths.data("Update/True/SMMNLP_training_true_phase3.xlsx")
true_all = pd.read_excel(truedata)

In [ ]:
true_all.head(5)

In [ ]:
len(true_all)

## Read in newly generated results


In [ ]:
readfileresults = paths.data("Update/ExtraLLMRun/phase3_fewshot_dxdate_human_updtraining.xlsx")
results = pd.read_excel(readfileresults)[
    ["PatientSSN", "NoteDate", "UncertainFlag_LLM", "DxDate_Human", "DxDate_LLM"]
]

In [ ]:
results.head()

In [ ]:
len(results)

## Combine true and result data for performance


In [ ]:
final = true_all.merge(results, on=["PatientSSN", "NoteDate"], how="inner")

In [ ]:
len(final)

In [ ]:
final.head(5)

## SMM Date Extraction Performance


In [ ]:
final["DxDate_Reviewed"] = np.where(
    final["UncertainFlag_LLM"] == True, final["DxDate_Human"], final["DxDate_LLM"]
)

In [ ]:
date_diff_human = final[
    (final["DxDate_Reviewed"] != "unknown") & (final["SMM Dx Date"] != "unknown")
]

In [ ]:
len(date_diff_human)

In [ ]:
date_diff_nohuman = final[(final["DxDate_LLM"] != "unknown") & (final["SMM Dx Date"] != "unknown")]

In [ ]:
len(date_diff_nohuman)

In [ ]:
date_diff_nohuman["DxDate_clean"] = date_diff_nohuman["DxDate_LLM"].apply(expand_partial_date)
date_diff_nohuman["HxDate"] = date_diff_nohuman["SMM Dx Date"].apply(expand_partial_date)

In [ ]:
date_diff_human["DxDate_clean"] = date_diff_human["DxDate_Reviewed"].apply(expand_partial_date)
date_diff_human["HxDate"] = date_diff_human["SMM Dx Date"].apply(expand_partial_date)

In [ ]:
date_diff_nohuman["True_SMM_date"] = pd.to_datetime(date_diff_nohuman["HxDate"], format="%m/%d/%Y")
date_diff_nohuman["LLM_SMM_date"] = pd.to_datetime(
    date_diff_nohuman["DxDate_clean"], format="%m/%d/%Y"
)

In [ ]:
date_diff_human["True_SMM_date"] = pd.to_datetime(date_diff_human["HxDate"], format="%m/%d/%Y")
date_diff_human["LLM_SMM_date"] = pd.to_datetime(date_diff_human["DxDate_clean"], format="%m/%d/%Y")

In [ ]:
date_diff_human["date_diff"] = date_diff_human["LLM_SMM_date"] - date_diff_human["True_SMM_date"]

In [ ]:
date_diff_nohuman["date_diff"] = (
    date_diff_nohuman["LLM_SMM_date"] - date_diff_nohuman["True_SMM_date"]
)

In [ ]:
max(date_diff_human["date_diff"])

In [ ]:
min(date_diff_human["date_diff"])

In [ ]:
max(date_diff_nohuman["date_diff"])

In [ ]:
min(date_diff_nohuman["date_diff"])